In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [6]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [7]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
# create subagents

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [10]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [11]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [12]:
from rich import print as rprint

rprint(response)

{
    'messages': [
        HumanMessage(
            content='What is the square root of 456?',
            additional_kwargs={},
            response_metadata={},
            id='dbcabe4e-20e0-4791-8acc-c2aa01a85b33'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 383,
                    'total_tokens': 414,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-9f8b3b4e4c20d640',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019d2838-6c0b-7923-b6a2-d75335c8b378-0',
            tool_calls=[
                {
                    'name': 'call_subagent_1',
                    'args': {'x': 456},
                    'id': 'chatcmpl-tool-bfd3c6be784f668d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 383,
                'output_tokens': 31,
                'total_tokens': 414,
                'input_token_details': {},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='The square root of 456.0 is approximately 21.354.',
            name='call_subagent_1',
            id='7495f99d-0dd7-4358-a7fb-e1b57bf94cc3',
            tool_call_id='chatcmpl-tool-bfd3c6be784f668d'
        ),
        AIMessage(
            content='The square root of 456 is approximately 21.354.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 19,
                    'prompt_tokens': 451,
                    'total_tokens': 470,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'Qwen3.5-397B-A17B-FP8',
                'system_fingerprint': None,
                'id': 'chatcmpl-9b068c70b2f91496',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019d2838-6f98-7721-9eb6-fde5738985b6-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 451,
                'output_tokens': 19,
                'total_tokens': 470,
                'input_token_details': {},
                'output_token_details': {}
            }
        )
    ]
}